#  Topic Discovery & Hierarchical Trend Analysis

This notebook demonstrates an end-to-end pipeline that automatically discovers
topics in customer complaints, validates their coherence, and organizes them
into a navigable hierarchy.

**Key principle:** topics are discovered from the text itself, no predefined
categories. This makes the method directly applicable to unlabelled client data.

**Validation dataset:** CFPB Consumer Complaints (labelled), used to prove the
method.

> **Core principle:** topics are discovered directly from the complaint text, with no
> predefined categories. This is what makes the method applicable to new, unlabelled
> client conversations.

<br>

## 1. Embeddings

Each complaint is converted into a numerical vector that captures its meaning, using
OpenAI `text-embedding-3-small` (1536 dimensions).

* Complaints about the same subject produce vectors that are close together.
* Closeness is measured by cosine similarity.
* Only the complaint text is embedded, not the date or any label.

<br>

## 2.  Dimensionality Reduction (UMAP)

The 1536 dimensions are compressed down to 5 while preserving the neighbourhood
structure.

* High dimensional vectors are hard to cluster directly (everything looks equally far).
* UMAP keeps similar complaints close while reducing dimensions.
* This makes the clustering step both faster and more reliable.

<br>

## 3.  Clustering (BERTopic with HDBSCAN)

Similar complaints are grouped into topics using HDBSCAN, the clustering engine
inside BERTopic.

* HDBSCAN groups dense regions of complaints into topics.
* The number of topics is discovered automatically and is **not fixed in advance**.
* The only setting is the minimum group size, which controls how fine grained the
  topics are.
* Complaints that do not fit any dense region are temporarily marked as outliers.

<br>

## 4.  Outlier Recovery

HDBSCAN leaves some complaints ungrouped. Each of these is reassigned to its nearest
topic based on embedding similarity, so no complaint is lost.

<br>

## 5.  Topic Centroids

For each topic we compute its centroid, which is the average of the embeddings of all
its complaints.

* The centroid represents the centre, or the most typical position, of a topic.
* It is used to find the complaints that best represent each topic.

<br>

## 6.  Topic Labelling (LLM)

For each topic, the complaints closest to its centroid are sent to a language model,
which produces a short, human readable label and a one sentence description.

* Input: the 10 complaints nearest the centroid (the most representative ones).
* Output: a clear label such as "Mortgage Payment Issues" plus a description.
* This turns abstract clusters into business readable topics.

<br>

## 7.  Validation (LLM as Judge)

We independently verify that each topic is coherent, producing a quality score
**without using any labels**. This is essential, because client data has no labels.

> **The process for each topic**
> 1. The model infers a category from 5 representative complaints.
> 2. It then scores 10 randomly chosen complaints (0 to 3) against that category.
> 3. The average score, normalized to a 0 to 1 range, is the coherence score.

* Random sampling tests the whole topic, not only the best examples.
* A high average means the topic is internally consistent.
* Because it needs no labels, it transfers directly to client data.

<br>

## 8.  Hierarchical Clustering (the core method)

The discovered topics are organized into a hierarchy that can be explored from broad
themes down to specific topics. The structure is computed entirely from the data using
agglomerative clustering with Ward linkage, the same family of method BERTopic uses.

> **How it works**
> 1. Each topic is represented by its centroid.
> 2. The two closest topics are merged into a parent group, then distances are recomputed.
> 3. This repeats until a single root remains, forming the tree.

>  **Why it is dynamic, not arbitrary**
> The number of levels is not fixed in advance. It emerges from how the topics relate
> to each other. The structure is objective and fully reproducible. The language model
> only adds the readable names on top of a structure that is entirely data driven.

<br>

## 9.  The End Result: an Interactive Treegraph

The hierarchy is displayed as an interactive treegraph.

* **Leaves** are the discovered topics, the most specific level.
* **Branches** are the broader categories that group related topics.
* **Root** represents all complaints combined.
* Each node can be expanded or collapsed, so the depth of view is fully controlled.

<br>

## 10.  Value and Use for Trend Analysis

* **One structure, many scopes:** the same tree gives both a high level overview and a
  detailed drill down.
* **Interpretable everywhere:** every node carries a readable label and description.
* **A foundation for trends:** because each node corresponds to a defined set of
  complaints, a trend can be measured at every node and read at any depth. A broad
  category may be rising overall while only some of its sub topics drive that increase,
  and the tree makes this visible by depth.

In [1]:
# PHASE 1: SETUP & MOUNT DRIVE

!pip install bertopic sentence-transformers datasets umap-learn hdbscan openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/trend_analysis_cfpb'
os.makedirs(PROJECT_DIR, exist_ok=True)


Mounted at /content/drive


## 1. Load the data

We load the public CFPB Consumer Complaints dataset, keeping the complaint text,
the date, and the official category labels (used only for later comparison).

- **Consumer complaint narrative**: the complaint narrative (the only input used for discovery)
- **Date received** : when the complaint was filed (used later for trends)
- **Issue**: official labels (used only to validate, never to cluster)

In [3]:
import pandas as pd

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
df = pd.read_csv(url, compression='zip', low_memory=False, nrows=200000)

# Garder et renommer les colonnes utiles
df = df.rename(columns={
    'Consumer complaint narrative': 'consumer_complaint_narrative',
    'Date received': 'date_received',
    'Product': 'product',
    'Issue': 'issue'
})
df = df[['consumer_complaint_narrative', 'date_received', 'product', 'issue']]

print(f"Total: {len(df):,}")
print(df['product'].value_counts())

Total: 200,000
product
Credit reporting or other personal consumer reports                             129640
Credit reporting, credit repair services, or other personal consumer reports     28731
Debt collection                                                                  14114
Mortgage                                                                          5993
Checking or savings account                                                       4751
Credit card                                                                       4128
Credit card or prepaid card                                                       2723
Money transfer, virtual currency, or money service                                2351
Credit reporting                                                                  1848
Student loan                                                                      1671
Vehicle loan or lease                                                             1192
Bank account or serv

In [4]:
df

,consumer_complaint_narrative,date_received,product,issue
0,NaN,2020-07-06,"Credit reporting, credit repair services, or o...",Incorrect information on your report
1,NaN,2019-12-26,Credit card or prepaid card,"Advertising and marketing, including promotion..."
2,These are not my accounts.,2020-05-08,"Credit reporting, credit repair services, or o...",Incorrect information on your report
3,Kindly address this issue on my credit report....,2024-01-05,Credit reporting or other personal consumer re...,Incorrect information on your report
4,NaN,2024-01-21,Credit reporting or other personal consumer re...,Improper use of your report
...,...,...,...,...
199995,Your agency is in violation of the FDCPA decep...,2025-12-08,Debt collection,Attempts to collect debt not owed
199996,NaN,2016-04-14,Mortgage,"Loan servicing, payments, escrow account"
199997,NaN,2024-04-18,Credit reporting or other personal consumer re...,Improper use of your report
199998,NaN,2026-03-10,Credit reporting or other personal consumer re...,Problem with a company's investigation into an...


In [18]:
import pandas as pd

# Keep only complaints with sufficiently long text
df = df[df['consumer_complaint_narrative'].notna()]
df = df[df['consumer_complaint_narrative'].str.len() >= 100]

# After filtering, before balancing and embeddings
before = len(df)
df = df.drop_duplicates(subset='consumer_complaint_narrative').reset_index(drop=True)
print(f"Doublons supprimes: {before - len(df)}")
print(f"Plaintes restantes: {len(df)}")

# Balance: maximum 2000 per product
TARGET = 2000
df_balanced = (
    df.groupby('product', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), TARGET), random_state=42))
      .reset_index(drop=True)
)

print(f"Balanced: {len(df_balanced):,}")
print(f"Products: {df_balanced['product'].nunique()}")

df_balanced.to_parquet(f'{PROJECT_DIR}/df_balanced.parquet', index=False)
print("Saved")

Doublons supprimes: 8218
Plaintes restantes: 39542
Balanced: 16,711
Products: 20


/tmp/ipykernel_2308/1807566123.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), TARGET), random_state=42))


Saved


## 3. Generate embeddings

Each complaint is converted into a numerical vector that captures its meaning.
Complaints about the same subject end up close together in this vector space.

- Model: OpenAI `text-embedding-3-small` (1536 dimensions)
- Similar complaints -> nearby vectors (measured by cosine similarity)
- Only the **Complaintes** is embedded

In [19]:
import os
import numpy as np
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('API_KEY'))

EMB_PATH = f'{PROJECT_DIR}/embeddings_openai.npy'


def get_openai_embeddings(texts, batch_size=64):
    """Compute OpenAI embeddings in batches"""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t)[:8000] for t in texts[i:i + batch_size]]
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch
        )
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"  {i + len(batch)}/{len(texts)}")
    return np.array(all_embeddings)


# Keep only valid string narratives
df_balanced = df_balanced[
    df_balanced['consumer_complaint_narrative'].apply(lambda x: isinstance(x, str))
].reset_index(drop=True)
print(f"df_balanced: {len(df_balanced)}")

# Compute embeddings on the text
embeddings = get_openai_embeddings(df_balanced['consumer_complaint_narrative'].tolist())
np.save(EMB_PATH, embeddings)
print(f"Embeddings: {embeddings.shape}")

assert len(embeddings) == len(df_balanced)
print("Aligned")

df_balanced: 16711
  64/16711
  128/16711
  192/16711
  256/16711
  320/16711
  384/16711
  448/16711
  512/16711
  576/16711
  640/16711
  704/16711
  768/16711
  832/16711
  896/16711
  960/16711
  1024/16711
  1088/16711
  1152/16711
  1216/16711
  1280/16711
  1344/16711
  1408/16711
  1472/16711
  1536/16711
  1600/16711
  1664/16711
  1728/16711
  1792/16711
  1856/16711
  1920/16711
  1984/16711
  2048/16711
  2112/16711
  2176/16711
  2240/16711
  2304/16711
  2368/16711
  2432/16711
  2496/16711
  2560/16711
  2624/16711
  2688/16711
  2752/16711
  2816/16711
  2880/16711
  2944/16711
  3008/16711
  3072/16711
  3136/16711
  3200/16711
  3264/16711
  3328/16711
  3392/16711
  3456/16711
  3520/16711
  3584/16711
  3648/16711
  3712/16711
  3776/16711
  3840/16711
  3904/16711
  3968/16711
  4032/16711
  4096/16711
  4160/16711
  4224/16711
  4288/16711
  4352/16711
  4416/16711
  4480/16711
  4544/16711
  4608/16711
  4672/16711
  4736/16711
  4800/16711
  4864/16711
  4928/16

## 4. Cluster the complaints (BERTopic)

We group similar complaints into topics using BERTopic, which combines two steps:

- **UMAP**: reduces the 1536 dimensions to 5 while preserving structure, so the
  clustering step works efficiently
- **HDBSCAN**: groups dense regions of complaints into topics

**Important:** the number of topics is **discovered automatically**, it is not
fixed in advance. The only setting is `min_cluster_size` (the minimum number of
complaints needed to form a topic). This is what makes the method dynamic.

In [29]:
import numpy as np
import time
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric='cosine', random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=80, min_samples=10, metric='euclidean',
    cluster_selection_method='eom', prediction_data=True
)

topic_model = BERTopic(
    umap_model=umap_model, hdbscan_model=hdbscan_model,
    nr_topics="auto",
    verbose=True, calculate_probabilities=False
)

print("Fitting BERTopic...")
start = time.time()
topics, _ = topic_model.fit_transform(
    df_balanced['consumer_complaint_narrative'].tolist(),
    embeddings
)
elapsed = (time.time() - start) / 60

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f"\nDone in {elapsed:.1f} min")
print(f"Topics: {n_topics} | Outliers: {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")

2026-05-22 15:06:18,836 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic...


2026-05-22 15:06:53,450 - BERTopic - Dimensionality - Completed ✓
2026-05-22 15:06:53,452 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-22 15:07:01,205 - BERTopic - Cluster - Completed ✓
2026-05-22 15:07:01,208 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-22 15:07:07,145 - BERTopic - Representation - Completed ✓
2026-05-22 15:07:07,148 - BERTopic - Topic reduction - Reducing number of topics
2026-05-22 15:07:07,178 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-22 15:07:10,264 - BERTopic - Representation - Completed ✓
2026-05-22 15:07:10,271 - BERTopic - Topic reduction - Reduced number of topics from 33 to 13



Done in 0.9 min
Topics: 12 | Outliers: 4194 (25.1%)


## 5. Recover outliers

HDBSCAN leaves some complaints ungrouped (outliers). We reassign each of them to
the most similar existing topic, so no complaint is lost.

- Strategy: assign each outlier to its nearest topic by embedding similarity
- Result: every complaint belongs to a topic

In [26]:
new_topics = topic_model.reduce_outliers(
    df_balanced['consumer_complaint_narrative'].tolist(),
    topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.3
)

topic_model.update_topics(
    df_balanced['consumer_complaint_narrative'].tolist(),
    topics=new_topics
)

final_topics = new_topics
n_outliers = sum(1 for t in final_topics if t == -1)
print(f"Outliers after reduction: {n_outliers} ({n_outliers/len(final_topics)*100:.1f}%)")

2026-05-22 15:02:27,751 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers after reduction: 0 (0.0%)


## 6. Compute topic centroids

For each topic we compute its **centroid**, the average position of all its
complaints. The centroid represents the "centre" of the topic.

- `topic_centroids: the centre vector of each topic
- `topic_indices`: which complaints belong to each topic
- Used to find the most representative complaints of each topic

In [27]:
import numpy as np

final_topics_arr = np.array(final_topics)

topic_centroids = {}
topic_indices = {}

for tid in set(final_topics):
    if tid == -1:
        continue
    mask = final_topics_arr == tid
    topic_centroids[tid] = embeddings[mask].mean(axis=0)
    topic_indices[tid] = np.where(mask)[0]

print(f"Centroids calculated for {len(topic_centroids)} topics")

Centroids calculated for 31 topics


## 7. Label each topic (LLM)

For each topic, we take the complaints closest to its centroid and ask a language
model to produce a short, human-readable label and description.

- Input: the 10 complaints nearest the centroid (most representative)
- Output: a clear label (e.g. "Mortgage Payment Issues") + one-sentence description
- This turns abstract clusters into business-readable topics

In [31]:
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_representative_docs(topic_id, n=10):
    centroid = topic_centroids[topic_id].reshape(1, -1)
    indices = topic_indices[topic_id]
    cluster_embs = embeddings[indices]
    cluster_texts = df_balanced.iloc[indices]['consumer_complaint_narrative'].tolist()
    sims = cosine_similarity(cluster_embs, centroid).flatten()
    top_idx = sims.argsort()[-n:][::-1]
    return [cluster_texts[i] for i in top_idx]

def label_cluster(topic_id, existing_labels):
    rep_docs = get_representative_docs(topic_id, n=10)
    docs_text = "\n\n".join([f"- {str(d)[:400]}" for d in rep_docs])

    used = "\n".join([f"- {lbl}" for lbl in existing_labels]) if existing_labels else "(none yet)"

    prompt = f"""Read these customer complaints from one cluster.

{docs_text}

Labels already assigned to other clusters:
{used}

Give this cluster a label that captures what makes it SPECIFIC and DISTINCT from the clusters above. Focus on the most precise, defining characteristic of these complaints (the company involved, the exact problem, or the product type).

Respond ONLY with valid JSON:
{{"label": "3-5 word label", "description": "one sentence"}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120, temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

topic_labels = {}
existing = []   # liste des labels deja attribues

for tid in sorted(topic_centroids.keys()):
    result = label_cluster(tid, existing)
    topic_labels[tid] = result
    existing.append(result['label'])   # ajouter le nouveau label a la liste
    print(f"Topic {tid}: {result['label']}")

import pandas as pd
pd.DataFrame([
    {'topic_id': k, 'label': v['label'], 'description': v['description']}
    for k, v in topic_labels.items()
]).to_csv(f'{PROJECT_DIR}/topic_labels.csv', index=False)
print("Saved")

Topic 0: Mortgage Payment Issues
Topic 1: Identity Theft Complaints
Topic 2: Fraudulent Transactions Complaints
Topic 3: Chase Fraud Alerts and Scams
Topic 4: Fair Credit Reporting Violations
Topic 5: Capital One Account Issues
Topic 6: Wells Fargo Unauthorized Charges
Topic 7: PayPal Account Security Issues
Topic 8: Harassment by Debt Collectors
Topic 9: Unauthorized Credit Inquiries
Topic 10: American Express Dispute Issues
Topic 11: Cash App Account Issues
Topic 12: Zelle Dispute Resolution Issues
Topic 13: Chime Bank Fraud Disputes
Topic 14: Synchrony Bank Payment Issues
Topic 15: Cash App Fraud and Dispute Issues
Topic 16: Navy Federal Overdraft Fees
Topic 17: Coinbase Account Access Issues
Topic 18: Credit Report Validation Issues
Topic 19: Prepaid Card Access Issues
Topic 20: Credit Report Dispute Delays
Topic 21: Discover Card Dispute Issues
Topic 22: PNC Bank Account Issues
Topic 23: Citibank Account Bonus Issues
Topic 24: Cash App Fraud Resolution Complaints
Topic 25: Mr. Coo

## 8. Validate topic coherence (LLM-as-Judge)

We independently verify that each topic is coherent, producing a quality score
**without using any labels**, crucial for unlabelled client data.

Inspired by the ProxAnn method (ACL 2025). For each topic:

1. **Infer** a category from 5 representative complaints
2. **Score** 10 randomly selected complaints (0–3) against that category
3. **Average** the scores -> a normalized coherence score (0 to 1)

- Random sampling tests the whole topic, not just the best examples
- A high average score means the topic is internally consistent
- Works without labels, so it transfers directly to client data

In [34]:
# SECTION 10: LLM AS A JUDGE

import json
import time
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('API_KEY'))


def proxann_infer_category(topic_id, n_docs=5):
    """STEP 1: LLM infers the cluster's category from top docs."""
    centroid = topic_centroids[topic_id].reshape(1, -1)
    indices = topic_indices[topic_id]
    cluster_embs = embeddings[indices]
    cluster_texts = df_balanced.iloc[indices]['consumer_complaint_narrative'].tolist()

    sims = cosine_similarity(cluster_embs, centroid).flatten()
    top_idx = sims.argsort()[-n_docs:][::-1]
    rep_docs = [str(cluster_texts[i])[:400] for i in top_idx]
    docs_text = "\n\n".join([f"- {d}" for d in rep_docs])

    prompt = f"""Read these {n_docs} customer complaints from the same cluster.

{docs_text}

What category best describes this cluster?

Respond ONLY with a valid JSON object:
{{
  "category": "3-7 word category label"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)['category']


def proxann_fit_score(category, document):
    """STEP 2: LLM scores how well a document fits the category (0-3)."""
    prompt = f"""Category: "{category}"

Document: "{str(document)[:400]}"

Rate how well this document fits the category:
- 3: Perfect fit (clearly about this category)
- 2: Good fit (mostly about this category)
- 1: Partial fit (touches on it but mainly about something else)
- 0: No fit (not about this category)

Respond ONLY with a valid JSON object:
{{
  "score": <integer 0, 1, 2, or 3>
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0,
        response_format={"type": "json_object"}
    )
    return int(json.loads(response.choices[0].message.content)['score'])


def evaluate_cluster_proxann(topic_id, n_eval_docs=10):
    """Run full ProxAnn evaluation for one cluster."""
    # Step 1: Category inference
    category = proxann_infer_category(topic_id)

    # Step 2: Random sample for fit evaluation
    indices = topic_indices[topic_id]
    sample_size = min(n_eval_docs, len(indices))
    sample_indices = np.random.choice(indices, sample_size, replace=False)

    fit_scores = []
    for idx in sample_indices:
        doc = df_balanced.iloc[idx]['consumer_complaint_narrative']
        try:
            score = proxann_fit_score(category, doc)
            fit_scores.append(score)
        except Exception as e:
            print(f"   Skipping one doc: {e}")
        time.sleep(0.3)

    return {
        'topic_id': int(topic_id),
        'inferred_category': category,
        'fit_scores': fit_scores,
        'mean_fit': float(np.mean(fit_scores)) if fit_scores else 0.0,
        'normalized_fit': float(np.mean(fit_scores) / 3.0) if fit_scores else 0.0,
        'n_evaluated': len(fit_scores)
    }


# Run on all topics
print(f"Each topic = 1 inference + 10 fit scores = 11 LLM calls\n")

proxann_results = {}

for i, topic_id in enumerate(sorted(topic_centroids.keys()), 1):
    try:
        result = evaluate_cluster_proxann(topic_id, n_eval_docs=10)
        proxann_results[topic_id] = result
        print(f"[{i:2}/{len(topic_centroids)}] Topic {topic_id:2}: "
              f"'{result['inferred_category']}' -> Fit: {result['normalized_fit']:.2f}")
    except Exception as e:
        print(f" Topic {topic_id}: {e}")
        proxann_results[topic_id] = None

# Global score
scores = [r['normalized_fit'] for r in proxann_results.values() if r is not None]
print(f"\nMean fit: {np.mean(scores):.2f} | Median: {np.median(scores):.2f}")

# Save results
with open(f'{PROJECT_DIR}/proxann_results.json', 'w') as f:
    json.dump(
        {str(k): v for k, v in proxann_results.items() if v is not None},
        f, indent=2
    )

print("ProxAnn results saved")

Each topic = 1 inference + 10 fit scores = 11 LLM calls

[ 1/31] Topic  0: 'Mortgage Payment and Modification Issues' -> Fit: 0.83
[ 2/31] Topic  1: 'Identity Theft and Credit Disputes' -> Fit: 0.90
[ 3/31] Topic  2: 'Fraud and Account Security Issues' -> Fit: 0.73
[ 4/31] Topic  3: 'Fraudulent Activity and Account Security' -> Fit: 0.80
[ 5/31] Topic  4: 'Fair Credit Reporting Act Violations' -> Fit: 1.00
[ 6/31] Topic  5: 'Fraud and Billing Disputes' -> Fit: 0.90
[ 7/31] Topic  6: 'Unauthorized Transactions and Fees' -> Fit: 0.90
[ 8/31] Topic  7: 'PayPal account access and fraud issues' -> Fit: 0.83
[ 9/31] Topic  8: 'Harassment and Miscommunication by Debt Collectors' -> Fit: 1.00
[10/31] Topic  9: 'Unauthorized Credit Inquiries Complaints' -> Fit: 1.00
[11/31] Topic 10: 'Customer Complaints about American Express Services' -> Fit: 0.93
[12/31] Topic 11: 'Cash App transaction disputes and scams' -> Fit: 0.73
[13/31] Topic 12: 'Zelle dispute handling complaints' -> Fit: 0.90
[14/31]

## 9. Hierarchical clustering
## Building the Topic Hierarchy

The discovered topics are organized into a hierarchy that can be explored from
broad themes down to specific topics. The structure is computed entirely from the
data, the number of levels is **not fixed in advance**, it emerges from the
topics themselves.

### Step 1: Represent each topic by its centroid

Each topic is summarized by its **centroid**: the average of the embedding vectors
of all complaints in that topic. The centroid is the topic's position in semantic
space, topics about similar issues have nearby centroids.

### Step 2: Build the tree with agglomerative clustering (Ward)

We apply **agglomerative hierarchical clustering** with **Ward linkage** on the
centroids. This is the same family of method BERTopic uses for its hierarchical
topics, and it is fully deterministic:

1. Compute the distance between every pair of topic centroids.
2. Merge the two closest groups into a new parent node.
3. Recompute distances and merge the next-closest pair.
4. Repeat until a single root remains.

Ward linkage merges the pair that minimizes the increase in within-group variance,
which produces compact, balanced groups. The complete sequence of merges forms a
**binary tree** (a dendrogram): the closer two topics are, the earlier they merge.

**This is dynamic:** no target number of clusters is imposed. The hierarchy and its
depth are a direct consequence of how the topics relate to each other.

### Step 3: Name each level (LLM)

The tree structure is data-driven, but the node names are made human-readable with
a language model. Labels are generated **bottom-up**: each merge node is named from
the labels of its two children, with the constraint that the parent name must be
broader than and distinct from its children. This avoids repeated names and gives
each level a meaningful, interpretable category.

### Step 4: Visualize as an interactive treegraph

The hierarchy is rendered as a **Highcharts treegraph**:

- **Leaves** are the discovered topics (most specific).
- **Internal nodes** are the broader categories produced by the clustering.
- **Root** represents all complaints.
- Each node shows the **share of complaints** it contains, and is
  **expandable / collapsible**, so the analyst controls the level of depth shown.

### Value

- **One structure, multiple scopes:** a high-level overview and a detailed
  drill-down come from the same tree.
- **Objective and reproducible:** the structure is computed from the data, the same input always yields the same tree.
- **A foundation for trend analysis:** because every node corresponds to a defined
  set of complaints, trends can be measured at each node and read at any depth.

In [67]:
import numpy as np
import json
from scipy.cluster.hierarchy import linkage
from collections import defaultdict

# 1. Centroid per topic
topic_ids = sorted(topic_labels.keys())
n = len(topic_ids)
centroids = np.array([topic_centroids[t] for t in topic_ids])

# 2. Build the FULL binary tree (Ward), dynamic
Z = linkage(centroids, method='ward')

# 3. Map scipy's tree into nodes
nodes = {}
for i, t in enumerate(topic_ids):
    nodes[i] = {'name': topic_labels[t]['label'],
                'description': topic_labels[t].get('description', ''),
                'parent': None, 'topic': t}
for k in range(Z.shape[0]):
    new_id = n + k
    c1, c2 = int(Z[k, 0]), int(Z[k, 1])
    nodes[new_id] = {'name': None, 'description': None, 'parent': None}
    nodes[c1]['parent'] = new_id
    nodes[c2]['parent'] = new_id
root_id = n + Z.shape[0] - 1
print(f"Full dynamic tree: {len(nodes)} nodes, root = {root_id}")

# 4. children map + leaf members
children = defaultdict(list)
for nid, node in nodes.items():
    if node['parent'] is not None:
        children[node['parent']].append(nid)

def leaf_members(nid):
    if 'topic' in nodes[nid]:
        return [nodes[nid]['topic']]
    out = []
    for c in children[nid]:
        out.extend(leaf_members(c))
    return out

# 5. Label each merge node from its TWO children
def label_from_children(child_names):
    kids = "\n".join([f"- {c}" for c in child_names])
    prompt = f"""Two groups are combined into one broader category:

{kids}

Give a BROADER parent category, DIFFERENT from both children.
It MUST NOT be identical to either child name.

Respond ONLY with valid JSON:
{{"label": "3-5 word broader category", "description": "one sentence"}}"""
    try:
        r = client.chat.completions.create(
            model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
            max_tokens=120, temperature=0, response_format={"type": "json_object"})
        return json.loads(r.choices[0].message.content)
    except Exception:
        return {"label": "Group", "description": ""}

for k in range(Z.shape[0]):
    new_id = n + k
    child_names = [nodes[c]['name'] for c in children[new_id]]
    res = label_from_children(child_names)
    nodes[new_id]['name'] = res['label']
    nodes[new_id]['description'] = res.get('description', '')
    print(f"Node {new_id}: {res['label']}")

# 6. Counts
final_topics_arr = np.array(final_topics)
total = len(final_topics_arr)
for nid, node in nodes.items():
    members = leaf_members(nid)
    c = int(np.isin(final_topics_arr, members).sum())
    node['count'], node['pct'] = c, round(c / total * 100, 1)

# 7. Highcharts data
data = []
for nid, node in nodes.items():
    pt = {'id': str(nid), 'name': f"{node['name']} ({node['pct']}%)",
          'description': str(node['description'] or ''), 'count': node['count']}
    if node['parent'] is not None:
        pt['parent'] = str(node['parent'])
    data.append(pt)
print(f"Data points: {len(data)}")

data_json = json.dumps(data, ensure_ascii=False).replace('<', '\\u003c')

# 8. HTML: no menu icon, auto-download PNG on open
html = """<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://cdn.jsdelivr.net/npm/highcharts@11.4.8/highcharts.js"></script>
<script src="https://cdn.jsdelivr.net/npm/highcharts@11.4.8/modules/treemap.js"></script>
<script src="https://cdn.jsdelivr.net/npm/highcharts@11.4.8/modules/treegraph.js"></script>
<script src="https://cdn.jsdelivr.net/npm/highcharts@11.4.8/modules/exporting.js"></script>
<script src="https://cdn.jsdelivr.net/npm/highcharts@11.4.8/modules/offline-exporting.js"></script>
<style>html,body{margin:0;font-family:sans-serif}#container{width:100%;height:1600px}</style>
</head><body>
<div id="container"></div>
<script>
window.onload = function () {
  var chart = Highcharts.chart('container', {
    chart: { marginLeft: 160, marginRight: 320, marginTop: 60 },
    title: { text: 'Hierarchie dynamique des topics (% du total)' },
    credits: { enabled: false },
    exporting: { enabled: false },
    tooltip: { useHTML: true, headerFormat: '',
      pointFormat: '<b>{point.name}</b><br>{point.count} plaintes<br><span style="font-size:11px">{point.description}</span>' },
    series: [{
      type: 'treegraph', data: __DATA__,
      marker: { radius: 7, fillColor: '#ffffff', lineWidth: 3 },
      dataLabels: { align: 'left', x: 14, pointFormat: '{point.name}',
        style: { fontSize: '10px', color: '#000', textOutline: '3px white', whiteSpace: 'nowrap' },
        crop: false, overflow: 'allow' },
      collapseButton: { onlyOnHover: false, shape: 'circle', height: 16, width: 16 },
      levels: [
        { level: 2, colorByPoint: true },
        { level: 3, colorVariation: { key: 'brightness', to: -0.4 } },
        { level: 4, colorVariation: { key: 'brightness', to: 0.4 } }
      ]
    }]
  });
  // Auto-download a PNG once the chart is rendered (no menu icon needed)
  setTimeout(function () {
    chart.exportChartLocal({ type: 'image/png', filename: 'topic_treegraph', scale: 2 });
  }, 1800);
};
</script></body></html>"""

html = html.replace('__DATA__', data_json)
with open(f'{PROJECT_DIR}/topic_treegraph.html', 'w', encoding='utf-8') as f:
    f.write(html)
print("Saved")

# Auto-download the HTML from Colab
from google.colab import files
files.download(f'{PROJECT_DIR}/topic_treegraph.html')

Full dynamic tree: 61 nodes, root = 60
Node 31: Cash App Security Concerns
Node 32: Financial Service Issues
Node 33: Fraudulent Financial Activities
Node 34: Financial Fraud and Issues
Node 35: Financial Distress and Harassment
Node 36: Digital Payment Concerns
Node 37: Financial Fraud Prevention
Node 38: Financial Integrity Issues
Node 39: Financial Institution Disputes
Node 40: Online Financial Safety
Node 41: Financial Compliance Challenges
Node 42: Banking and Financial Concerns
Node 43: Credit Card Dispute Management
Node 44: Banking Service Challenges
Node 45: Digital Finance Security Concerns
Node 46: Financial Risk Management
Node 47: Banking Customer Complaints
Node 48: Financial Services and Support
Node 49: Banking Service Issues
Node 50: Banking and Financial Services
Node 51: Mortgage and Financial Challenges
Node 52: Financial Security and Integrity
Node 53: Consumer Financial Compliance
Node 54: Financial Systems and Security
Node 55: Financial Services Management
Node 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Trend analysis

For each topic (and each branch of the tree), we measure whether complaints are
rising or falling over time.

- **Mann-Kendall test**: a standard non-parametric test for detecting trends
- Reports the direction (increasing / decreasing / stable), statistical
  significance (p-value), and rate of change (slope)
- Applied per node, this turns the hierarchy into a navigable trend map

In [17]:
!pip install pymannkendall -q


In [18]:
import pandas as pd
import pymannkendall as mk

docs = df_balanced['consumer_complaint_narrative'].tolist()
timestamps = pd.to_datetime(df_balanced['date_received']).tolist()

# Topics au fil du temps
topics_over_time = topic_model.topics_over_time(docs, timestamps, nr_bins=20)

# Mann-Kendall par topic
trend_results = []
for topic_id in sorted(topics_over_time['Topic'].unique()):
    if topic_id == -1:
        continue
    topic_data = topics_over_time[topics_over_time['Topic'] == topic_id].sort_values('Timestamp')
    frequencies = topic_data['Frequency'].values
    if len(frequencies) < 3:
        continue
    result = mk.original_test(frequencies)
    trend_results.append({
        'topic_id': topic_id,
        'label': topic_labels[topic_id]['label'] if topic_id in topic_labels else '?',
        'trend': result.trend,
        'p_value': round(result.p, 4),
        'significant': result.p < 0.05,
        'slope': round(result.slope, 2)
    })

trend_df = pd.DataFrame(trend_results).sort_values('slope', ascending=False)
print(trend_df.to_string(index=False))
trend_df.to_csv(f'{PROJECT_DIR}/trend_analysis.csv', index=False)

20it [00:18,  1.08it/s]


 topic_id                                label      trend  p_value  significant  slope
        0              Identity Theft Disputes increasing   0.0000         True  25.07
        1   Fraudulent Transactions Complaints increasing   0.0000         True  17.38
        8 Fair Credit Reporting Act Violations increasing   0.0001         True   5.92
        3           Loan Issues and Complaints increasing   0.0000         True   2.89
        5              Chase Bank Fraud Issues increasing   0.0000         True   2.80
        7             Fraud and Account Issues increasing   0.0001         True   1.89
       12   Cash App Unauthorized Transactions increasing   0.0000         True   1.65
        4                Loan Servicing Issues   no trend   0.1350        False   1.44
        6      Wells Fargo Customer Complaints increasing   0.0090         True   1.07
       10                PayPal Account Issues increasing   0.0000         True   1.00
        9        Zelle Dispute Handling Iss

In [21]:
import pandas as pd
import plotly.express as px

tot_labeled = topics_over_time.copy()
tot_labeled['Sujet'] = tot_labeled['Topic'].map(
    lambda t: topic_labels[t]['label'] if t in topic_labels else f'Topic {t}'
)
tot_labeled = tot_labeled[tot_labeled['Topic'] != -1]

fig = px.line(
    tot_labeled, x='Timestamp', y='Frequency', color='Sujet',
    title='Evolution of Complaint Topics (2015–2026)', markers=True
)
fig.update_layout(width=1300, height=800)
fig.write_html(f'{PROJECT_DIR}/trends_all.html')
fig.show()